In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Transformer --> Encoder 

In [1]:
import torch 
import torch.nn as nn
import math 

## Token Embeddings

In [2]:
class TokenEmbedding(nn.Module):

    def __init__(self, vocab_size:int, d_model:int):   # vocab_size --> rows, d_model --> cols
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self,x): # scaling the token embeddings by sqrt of d_model
        return self.embedding(x) * math.sqrt(self.d_model)


### position embedding

In [3]:
class PosotionalEncoding(nn.Module):

    def __init__(self, d_model:int, max_len:int = 5000, dropout:float=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        
        pe[:,0::2] = torch.sin(position * div_term) # for even index
        pe[:,1::2] = torch.cos(position * div_term) # for odd index
        pe = pe.unsqueeze(0)  # shape--> [1,max_len,d_model]
        self.register_buffer('pe',pe)  # saved in state_dict but not trained

    def forward(self,x):
        x = x + self.pe[:, :x.size(1)]  # x shape: [batch, seq_len, d_model]
        return self.dropout(x)
        

## Scaled Dot-Product Attention

In [4]:
class ScaledDotproductAttention(nn.Module):
    def __init__(self, dropout:float=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        # Q --> [batch, heads, seq, d_k]
        # k --> [batch, heads, seq, d_k]
        # v --> [batch, heads, seq, d_v]
        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2,-1))/(math.sqrt(d_k))

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        att_weight = torch.softmax(scores, dim=-1)
        att_weight = self.dropout(att_weight)

        output = torch.matmul(att_weight, V)

        return output, att_weight

## Multihead attention

In [5]:
class MultiheadAttention(nn.Module):
    def __init__(self, d_model:int, num_heads:int, dropout:float=0.1):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_k = d_model//num_heads 

        self.W_q = nn.Linear(d_model,d_model)
        self.W_k = nn.Linear(d_model,d_model)
        self.W_v = nn.Linear(d_model,d_model)
        self.W_o = nn.Linear(d_model,d_model)

        self.attention = ScaledDotproductAttention(dropout)
        self.dropout = nn.Dropout(dropout)

    def split_head(self, x, batch_size):
        x = x.view(batch_size,-1,self.num_heads,self.d_k)
        return x.transpose(1,2)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        Q = self.split_head(self.W_q(Q), batch_size)
        K = self.split_head(self.W_k(K), batch_size)
        V = self.split_head(self.W_v(V), batch_size)

        attn_output,_ = self.attention(Q,K,V,mask)

        #concatenate
        attn_output = attn_output.transpose(1,2).contiguous()
        attn_output = attn_output.view(batch_size, -1, self.num_heads * self.d_k)

        return self.W_o(attn_output)

        
        
        

## forward Network

In [6]:
class FeedForwordNetwork(nn.Module):
     def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)    
        self.linear2 = nn.Linear(d_ff, d_model)  
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()  # smoother than ReLU
        
     def forward(self, x):
         return self.linear2(self.dropout(self.activation(self.linear1(x))))



## transformer block

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model:int, num_head:int, d_ff:int, dropout:float=0.1):
        super().__init__()
        self.attention = MultiheadAttention(d_model, num_head, dropout)
        self.ffn = FeedForwordNetwork(d_model, d_ff, dropout)
        self.Lnorm1 = nn.LayerNorm(d_model)
        self.Lnorm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out = self.attention(self.Lnorm1(x),
                                  self.Lnorm1(x),
                                  self.Lnorm1(x), mask 
                                 )
        x = x + self.dropout(attn_out)


        ffn_out = self.ffn(self.Lnorm2(x))
        x = x + self.dropout(ffn_out)
        return x

In [8]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, num_layers=4, d_ff=1024, max_len=512, dropout=0.1):
        super().__init__()
        self.token_emb = TokenEmbedding(vocab_size, d_model)
        self.pos_enc = PosotionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.pos_enc(self.token_emb(x))
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [9]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model=256,
                 num_heads=8, num_layers=4, d_ff=1024,
                 max_len=512, dropout=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(
            vocab_size, d_model, num_heads,
            num_layers, d_ff, max_len, dropout
        )
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask=None):
       
        mask = None
        if attention_mask is not None:
           
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
        encoded = self.encoder(input_ids, mask)  
        cls_repr = encoded[:, 0, :]              
        return self.classifier(self.dropout(cls_repr))

In [10]:
model = TransformerClassifier(
    vocab_size = 30522,  # Bert vocab size
    num_classes = 2,  # for positive and negative
    d_model = 256, # embedding dim
    num_heads = 8, # attention heads
    num_layers = 4, #transformer layers
    d_ff = 1024, #feed forward hidden size
)

#### parameter count

In [ ]:
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}')


In [ ]:
batch_size, seq_len = 4, 128
dummy_ids  = torch.randint(0, 30522, (batch_size, seq_len))
dummy_mask = torch.ones(batch_size, seq_len, dtype=torch.long)
logits = model(dummy_ids, dummy_mask) 
print('Output shape:', logits.shape)   

### Traning

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
model     = model.to(device)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)


we need to dataset for training 

In [ ]:
for epoch in range(5):
    model.train()
    total_loss = 0
    for batch in train_loader:
        
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        logits = model(input_ids, attn_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        
    scheduler.step()
    print(f'Epoch {epoch+1}  Loss: {total_loss/len(train_loader):.4f}')

# for evaluating

In [ ]:
 model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            preds = model(input_ids, attn_mask).argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    print(f'Val Accuracy: {correct/total:.4f}')